Смена модели для rta

In [1]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.collection import Collection
from pymongo.database import Database

from utils.constants import RTA_MODEL


def get_mongo_client() -> MongoClient:
    """
    Создает и возвращает подключение к MongoDB на основе переменных окружения.
    """
    # Загрузка переменных окружения из файла .env
    load_dotenv()

    # Получение деталей подключения из переменных окружения
    mongo_username = os.getenv("MONGO_INITDB_ROOT_USERNAME")
    mongo_password = os.getenv("MONGO_INITDB_ROOT_PASSWORD")
    mongo_host = os.getenv("MONGO_HOST")
    mongo_port = os.getenv("MONGO_INITDB_ROOT_PORT")

    # Формирование URI для подключения к MongoDB
    mongo_uri = f"mongodb://{mongo_username}:{mongo_password}@{mongo_host}:{mongo_port}"

    return MongoClient(mongo_uri)


def update_model_in_rta(db: Database, new_model: str, excluded_statuses: list) -> None:
    """
    Обновляет значение поля model на заданное во всех документах коллекции RtA,
    у которых status не входит в excluded_statuses.

    Args:
        db (Database): Экземпляр базы данных MongoDB.
        new_model (str): Новое значение для поля model.
        excluded_statuses (list): Список статусов, при которых документы не будут обновлены.
    """
    collection: Collection = db["RtA"]
    query = {"status": {"$nin": excluded_statuses}}
    update = {"$set": {"model": new_model}}

    result = collection.update_many(query, update)
    print(f"Обновлено {result.modified_count} документов в коллекции RtA.")


def main() -> None:
    """
    Основная функция для обновления поля model в коллекции RtA для задач,
    у которых статус не входит в исключённые статусы.
    """
    # Подключение к MongoDB
    client = get_mongo_client()
    db = client["TrustLLM_ru"]

    # Новое значение поля model
    new_model = RTA_MODEL

    # Список статусов, при которых документы не будут обновлены
    excluded_statuses = ["completed", "measured"]

    # Обновление документов в RtA, у которых статус не в excluded_statuses
    update_model_in_rta(db, new_model, excluded_statuses)


main()


Обновлено 41767 документов в коллекции RtA.
